# Step 5b — Manual Coding Widget

Interactive widget for coding biographical field values from LLM-extracted text.

For each person and each field, you see all extracted quotes side-by-side with an input box to assign a coded value.

**Workflow:**
1. Select a person from the dropdown
2. For each field that has extracted text, read the quotes
3. Type your coded value in the input box (e.g., "Republican", "Yes", "1842", "Harvard")
4. Click Save or press Enter — value is written to the DB immediately
5. Use the navigation buttons to move between persons

In [11]:
import sys, json
from pathlib import Path
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

sys.path.insert(0, str(Path('.')))
from db import get_connection, save_coded_value, save_manual_coded_value

conn = get_connection()

FIELDS = [
    "political_affiliation",
    "wealthy_family",
    "birthplace",
    "ethnicity",
    "religion",
    "year_of_birth",
    "year_of_death",
    "education_level",
    "military_service",
    "public_office_sought",
    "railroad_stockholder",
    "railroad_board_member",
    "railroad_company_owner",
    "railroad_professional_services",
    "family_connection_railroad",
    "other_corporate_directorships",
    "real_estate",
    "wealth_at_death",
    "founded",
    "papers_owned_count",
    "internal_labor_struggles",
]

FIELD_LABELS = {f: f.replace('_', ' ').title() for f in FIELDS}

print("Ready.")

Ready.


## Progress overview

In [12]:
# Persons with extractions
persons_with_data = conn.execute("""
    SELECT p.research_id, p.canonical_name,
           COUNT(fe.id) as total_extractions,
           SUM(CASE WHEN fe.coded_value IS NOT NULL THEN 1 ELSE 0 END) as coded_count
    FROM persons p
    JOIN field_extractions fe ON p.research_id = fe.research_id
    GROUP BY p.research_id
    ORDER BY p.canonical_name
""").fetchall()

total_persons = len(persons_with_data)
fully_coded = sum(1 for p in persons_with_data if p['coded_count'] == p['total_extractions'])
partially_coded = sum(1 for p in persons_with_data if 0 < p['coded_count'] < p['total_extractions'])
uncoded = sum(1 for p in persons_with_data if p['coded_count'] == 0)

total_extractions = sum(p['total_extractions'] for p in persons_with_data)
total_coded = sum(p['coded_count'] for p in persons_with_data)

print(f"Persons with extractions: {total_persons}")
print(f"  Fully coded:    {fully_coded}")
print(f"  Partially coded: {partially_coded}")
print(f"  Not started:    {uncoded}")
print(f"\nExtractions: {total_coded}/{total_extractions} coded ({100*total_coded/max(total_extractions,1):.0f}%)")

Persons with extractions: 178
  Fully coded:    5
  Partially coded: 7
  Not started:    166

Extractions: 80/1209 coded (7%)


## Coding widget

In [13]:
# Build person list for dropdown
person_options = [(f"{p['canonical_name']} ({p['coded_count']}/{p['total_extractions']})", p['research_id'])
                  for p in persons_with_data]

# --- Widgets ---
person_dropdown = widgets.Dropdown(
    options=person_options,
    description='Person:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '80px'}
)

filter_dropdown = widgets.Dropdown(
    options=[('All fields', 'all'), ('Uncoded only', 'uncoded'), ('Coded only', 'coded')],
    value='all',
    description='Show:',
    layout=widgets.Layout(width='250px'),
    style={'description_width': '50px'}
)

nav_prev = widgets.Button(description='\u25c0 Prev', layout=widgets.Layout(width='80px'))
nav_next = widgets.Button(description='Next \u25b6', layout=widgets.Layout(width='80px'))
nav_next_uncoded = widgets.Button(description='Next uncoded \u25b6\u25b6', layout=widgets.Layout(width='130px'),
                                  button_style='info')

# Use a VBox container instead of Output widget — setting .children is atomic, no clear_output issues
content_box = widgets.VBox(layout=widgets.Layout(width='100%', min_height='400px'))
status_label = widgets.HTML(value='')

# Track active input widgets so we can save them
active_inputs = {}  # field_name -> (widget, extraction_ids)


def get_extractions_for_person(research_id):
    """Get all field extractions grouped by field for a person."""
    rows = conn.execute("""
        SELECT id, field_name, extracted_text, source_ref, confidence,
               identity_confidence, coded_value
        FROM field_extractions
        WHERE research_id = ?
        ORDER BY field_name, id
    """, (research_id,)).fetchall()

    grouped = {}
    for row in rows:
        fn = row['field_name']
        if fn not in grouped:
            grouped[fn] = []
        grouped[fn].append(dict(row))
    return grouped


def _build_field_with_evidence(field, extractions, filter_mode, research_id):
    """Build widgets for a field that has LLM-extracted evidence. Returns list of widgets or None."""
    existing_value = extractions[0]['coded_value']
    is_coded = existing_value is not None

    if filter_mode == 'uncoded' and is_coded:
        return None
    if filter_mode == 'coded' and not is_coded:
        return None

    label = FIELD_LABELS[field]

    # Build quotes HTML (skip [manual] placeholder rows)
    real_extractions = [e for e in extractions if e['extracted_text'] != '[manual]']
    quotes_html = ""
    for ext in real_extractions:
        conf_badge = ext['confidence']
        id_badge = f"identity: {ext['identity_confidence']}"
        source = ext['source_ref'] or 'unknown source'
        text = ext['extracted_text'].replace('<', '&lt;').replace('>', '&gt;')
        quotes_html += f"""
        <div style='background:#f8f8f0; border-left:3px solid #4a90d9; padding:6px 10px;
                    margin:4px 0; font-size:13px'>
            <span style='color:#888; font-size:11px'>[{conf_badge}, {id_badge}] {source}</span><br>
            <span style='font-style:italic'>\"{text}\"</span>
        </div>"""

    quote_count = len(real_extractions)
    header_html = widgets.HTML(f"""
    <div style='margin:10px 0 2px 0; padding:8px; border:1px solid #ddd; border-radius:4px;
                background: {"#f0fff0" if is_coded else "#fff"}'>
        <strong style='font-size:14px'>{label}</strong>
        <span style='color:#888; font-size:12px'>({quote_count} quote{"s" if quote_count != 1 else ""})</span>
        {quotes_html}
    </div>
    """)

    # Input + save button
    extraction_ids = [e['id'] for e in extractions]
    inp = widgets.Text(
        value=existing_value or '',
        placeholder=f'Code value for {label}...',
        layout=widgets.Layout(width='400px'),
    )
    save_btn = widgets.Button(description='Save', button_style='success',
                              layout=widgets.Layout(width='70px'))
    saved_label = widgets.HTML(value='\u2705 saved' if is_coded else '')

    def make_save_handler(field_name, ext_ids, inp_widget, lbl_widget):
        def handler(btn=None):
            val = inp_widget.value.strip()
            if not val:
                lbl_widget.value = '<span style="color:orange">empty \u2014 not saved</span>'
                return
            for eid in ext_ids:
                save_coded_value(conn, eid, val)
            lbl_widget.value = '\u2705 saved'
            status_label.value = f'<span style="color:green">Saved {field_name} = \"{val}\"</span>'
        return handler

    handler = make_save_handler(field, extraction_ids, inp, saved_label)
    save_btn.on_click(handler)
    inp.on_submit(lambda w, h=handler: h())

    active_inputs[field] = (inp, extraction_ids)
    return [header_html, widgets.HBox([inp, save_btn, saved_label])]


def _build_field_manual_only(field, research_id, filter_mode):
    """Build widgets for a field with no LLM evidence. Returns list of widgets or None."""
    label = FIELD_LABELS[field]

    existing = conn.execute(
        "SELECT id, coded_value FROM field_extractions WHERE research_id=? AND field_name=? AND extracted_text='[manual]'",
        (research_id, field)
    ).fetchone()
    existing_value = existing['coded_value'] if existing else None
    is_coded = existing_value is not None

    if filter_mode == 'uncoded' and is_coded:
        return None
    if filter_mode == 'coded' and not is_coded:
        return None

    header_html = widgets.HTML(f"""
    <div style='margin:4px 0 2px 0; padding:6px 8px; border:1px solid #eee; border-radius:4px;
                background: {"#f0fff0" if is_coded else "#fafafa"}'>
        <strong style='font-size:13px; color:#666'>{label}</strong>
        <span style='color:#aaa; font-size:11px'>(no extracted evidence)</span>
    </div>
    """)

    inp = widgets.Text(
        value=existing_value or '',
        placeholder=f'{label}...',
        layout=widgets.Layout(width='400px'),
    )
    save_btn = widgets.Button(description='Save', button_style='success',
                              layout=widgets.Layout(width='70px'))
    saved_label = widgets.HTML(value='\u2705 saved' if is_coded else '')

    def make_manual_handler(field_name, inp_widget, lbl_widget, rid):
        def handler(btn=None):
            val = inp_widget.value.strip()
            if not val:
                lbl_widget.value = '<span style="color:orange">empty \u2014 not saved</span>'
                return
            save_manual_coded_value(conn, rid, field_name, val)
            lbl_widget.value = '\u2705 saved'
            status_label.value = f'<span style="color:green">Saved {field_name} = \"{val}\"</span>'
        return handler

    handler = make_manual_handler(field, inp, saved_label, research_id)
    save_btn.on_click(handler)
    inp.on_submit(lambda w, h=handler: h())
    return [header_html, widgets.HBox([inp, save_btn, saved_label])]


def render_person(change=None):
    """Build the coding interface for the selected person and set it as content_box children."""
    active_inputs.clear()
    research_id = person_dropdown.value
    filter_mode = filter_dropdown.value

    person = conn.execute(
        "SELECT * FROM persons WHERE research_id=?", (research_id,)
    ).fetchone()

    grouped = get_extractions_for_person(research_id)

    children = []

    # Person header
    states = json.loads(person['known_states']) if person['known_states'] else []
    papers = json.loads(person['known_newspapers']) if person['known_newspapers'] else []
    children.append(widgets.HTML(f"""
    <h3 style='margin-bottom:5px'>{person['canonical_name']}</h3>
    <p style='color:#666; margin-top:0'>
        Active: {person['first_active_year'] or '?'}\u2013{person['last_active_year'] or '?'}
        &nbsp;|&nbsp; States: {', '.join(states) or '?'}
        &nbsp;|&nbsp; Papers: {', '.join(papers[:3]) or '?'}
    </p><hr>
    """))

    # --- Fields WITH extracted evidence ---
    fields_with_evidence = [f for f in FIELDS if f in grouped]
    fields_without_evidence = [f for f in FIELDS if f not in grouped]

    evidence_shown = 0
    for field in fields_with_evidence:
        field_widgets = _build_field_with_evidence(field, grouped[field], filter_mode, research_id)
        if field_widgets:
            children.extend(field_widgets)
            evidence_shown += 1

    # --- Separator + fields WITHOUT evidence ---
    manual_shown = 0
    for field in fields_without_evidence:
        field_widgets = _build_field_manual_only(field, research_id, filter_mode)
        if field_widgets:
            if manual_shown == 0:
                children.append(widgets.HTML("""
                <div style='margin:20px 0 10px 0; border-top:2px solid #ccc; padding-top:10px'>
                    <strong style='color:#888'>Other fields (no extracted evidence)</strong>
                </div>
                """))
            children.extend(field_widgets)
            manual_shown += 1

    if evidence_shown == 0 and manual_shown == 0:
        children.append(widgets.HTML(f"<p><em>No {filter_mode} fields for this person.</em></p>"))

    # Atomic update — replaces all children at once, no duplication possible
    content_box.children = children


def nav_prev_click(btn):
    idx = person_dropdown.index
    if idx > 0:
        person_dropdown.index = idx - 1

def nav_next_click(btn):
    idx = person_dropdown.index
    if idx < len(person_options) - 1:
        person_dropdown.index = idx + 1

def nav_next_uncoded_click(btn):
    current_idx = person_dropdown.index
    for i in range(current_idx + 1, len(person_options)):
        rid = person_options[i][1]
        uncoded = conn.execute("""
            SELECT COUNT(*) as n FROM field_extractions
            WHERE research_id=? AND coded_value IS NULL
        """, (rid,)).fetchone()['n']
        if uncoded > 0:
            person_dropdown.index = i
            return
    status_label.value = '<span style="color:blue">No more uncoded persons after this one!</span>'


nav_prev.on_click(nav_prev_click)
nav_next.on_click(nav_next_click)
nav_next_uncoded.on_click(nav_next_uncoded_click)

# Initial render before registering observers
render_person()

person_dropdown.observe(render_person, names='value')
filter_dropdown.observe(render_person, names='value')

# Single display call — the only thing this cell outputs
widgets.VBox([
    widgets.HBox([person_dropdown, filter_dropdown]),
    widgets.HBox([nav_prev, nav_next, nav_next_uncoded, status_label]),
    content_box
])

C:\Users\samwt\AppData\Local\Temp\ipykernel_40824\2963276540.py:115: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  inp.on_submit(lambda w, h=handler: h())
C:\Users\samwt\AppData\Local\Temp\ipykernel_40824\2963276540.py:167: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  inp.on_submit(lambda w, h=handler: h())


## Export coded data

In [14]:
import pandas as pd

# Pivot field extractions into one row per person
coded = conn.execute("""
    SELECT p.research_id, p.canonical_name, p.person_id,
           fe.field_name, fe.coded_value
    FROM field_extractions fe
    JOIN persons p ON fe.research_id = p.research_id
    WHERE fe.coded_value IS NOT NULL
    ORDER BY p.research_id, fe.field_name
""").fetchall()

if not coded:
    print("No coded values yet.")
else:
    df = pd.DataFrame([dict(r) for r in coded])
    # If multiple extractions per field, take first coded value
    df = df.drop_duplicates(subset=['research_id', 'field_name'], keep='first')
    pivot = df.pivot(index=['research_id', 'canonical_name', 'person_id'],
                     columns='field_name', values='coded_value').reset_index()
    
    out_path = Path('../data/personnel_coding/coded_biographies.csv')
    pivot.to_csv(out_path, index=False)
    print(f"Exported {len(pivot)} persons to {out_path}")
    display(pivot.head(10))

Exported 12 persons to ..\data\personnel_coding\coded_biographies.csv


field_name,research_id,canonical_name,person_id,birthplace,education_level,family_connection_railroad,founded,military_service,other_corporate_directorships,papers_owned_count,political_affiliation,public_office_sought,railroad_board_member,railroad_company_owner,railroad_professional_services,railroad_stockholder,wealthy_family,year_of_birth,year_of_death
0,4,Bayard Taylor,4,Pennsylvania,1,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,1825,1878
1,29,Berry R. Sulgrove,37,NaN,NaN,0,NaN,NaN,0,NaN,Republican,NaN,0,0,NaN,0,NaN,NaN,NaN
2,34,Buckley B. Paddock,44,NaN,NaN,NaN,0,1,NaN,1,NaN,1,1,0,NaN,1,NaN,NaN,NaN
3,37,Alfred Doten,48,NaN,NaN,0,NaN,NaN,Heavy investment in Mining,1,NaN,NaN,0,0,NaN,0,NaN,NaN,1903
4,66,A. J. Steinman,103,NaN,2,NaN,NaN,NaN,NaN,1,Democrat,1,NaN,NaN,NaN,1,NaN,1836,1917
5,130,Albert L. Fontaine,179,NaN,NaN,0,NaN,NaN,NaN,1,NaN,NaN,0,0,NaN,0,NaN,NaN,NaN
6,220,C. S. Douglas,285,NaN,1,0,NaN,NaN,NaN,NaN,NaN,1,0,0,NaN,0,NaN,NaN,NaN
7,280,A. P. Miller,373,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,291,C. W. Willard,382,NaN,NaN,0,NaN,NaN,NaN,NaN,Republican,1,0,0,1,0,NaN,NaN,NaN
9,354,Alfales Young,466,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,0,1,1853,1920
